# Netflix Movie Recommendation System

Reconstructed from the 47-page PDF supplied in this chat. The notebook follows the original project: TMDB movies + credits → feature extraction → text preprocessing/stemming → Bag of Words → cosine similarity → top-5 movie recommendations.

**Required input files**
- `tmdb_5000_movies.csv`
- `tmdb_5000_credits.csv`

Upload both CSV files when prompted in the next cell. This notebook is Google Colab-ready.


In [ ]:
# Install/import dependencies
!pip -q install nltk scikit-learn pandas numpy

import re
import ast
import pickle
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import nltk

from nltk.stem.porter import PorterStemmer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")


In [ ]:
# Upload the two TMDB CSV datasets to Colab
from google.colab import files

uploaded = files.upload()

required_files = {"tmdb_5000_movies.csv", "tmdb_5000_credits.csv"}
missing_files = required_files.difference(uploaded.keys())

if missing_files:
    raise FileNotFoundError(
        "Please upload both required files: " + ", ".join(sorted(missing_files))
    )

movies = pd.read_csv("tmdb_5000_movies.csv")
credits = pd.read_csv("tmdb_5000_credits.csv")

print("Movies shape:", movies.shape)
print("Credits shape:", credits.shape)
display(movies.head(3))
display(credits.head(3))


In [ ]:
# Merge the dataframes using the movie title (as in the original project)
movies = movies.merge(credits, on="title")
print("Merged shape:", movies.shape)
display(movies.head(2))


## 1. Select the relevant features

The original notebook selected `movie_id`, `title`, `overview`, `genres`, `keywords`, `cast`, and `crew`.


In [ ]:
df = movies[["movie_id", "title", "overview", "genres", "keywords", "cast", "crew"]].copy()
print("Selected dataframe shape:", df.shape)
display(df.head())


In [ ]:
# Convert JSON-like strings into Python lists
def fetch_names(text):
    """Extract each item's 'name' field from a JSON-like string."""
    if pd.isna(text):
        return []
    try:
        return [item["name"] for item in ast.literal_eval(text) if isinstance(item, dict) and "name" in item]
    except (ValueError, SyntaxError, TypeError):
        return []

def fetch_genres(text):
    return fetch_names(text)

def fetch_keywords(text):
    return fetch_names(text)

df["genres"] = df["genres"].apply(fetch_genres)
df["keywords"] = df["keywords"].apply(fetch_keywords)

display(df[["title", "genres", "keywords"]].head(3))


In [ ]:
# Extract the first three cast members
def fetch_cast(text):
    if pd.isna(text):
        return []
    try:
        cast_list = ast.literal_eval(text)
        return [person["name"] for person in cast_list[:3] if isinstance(person, dict) and "name" in person]
    except (ValueError, SyntaxError, TypeError):
        return []

df["cast"] = df["cast"].apply(fetch_cast)
display(df[["title", "cast"]].head(3))


In [ ]:
# Extract director name(s) from crew
def fetch_director(text):
    if pd.isna(text):
        return []
    try:
        crew_list = ast.literal_eval(text)
        return [
            person["name"]
            for person in crew_list
            if isinstance(person, dict) and person.get("job") == "Director" and "name" in person
        ]
    except (ValueError, SyntaxError, TypeError):
        return []

df["crew"] = df["crew"].apply(fetch_director)
display(df[["title", "crew"]].head(3))


In [ ]:
# Remove rows with missing values (the original notebook dropped null rows)
print("Missing values before cleaning:")
display(df.isnull().sum())

df.dropna(subset=["movie_id", "title", "overview", "genres", "keywords", "cast", "crew"], inplace=True)
df.reset_index(drop=True, inplace=True)

print("Missing values after cleaning:")
display(df.isnull().sum())


In [ ]:
# Convert overview text into a list of words
df["overview"] = df["overview"].apply(lambda text: text.split() if isinstance(text, str) else [])

# Combine overview, genres, keywords, cast and director into one tags column
df["tags"] = (
    df["overview"]
    + df["genres"]
    + df["keywords"]
    + df["cast"]
    + df["crew"]
)

data = df[["movie_id", "title", "tags"]].copy()
display(data.head())
print("Number of movies:", len(data))


In [ ]:
# Remove spaces inside individual tag phrases, matching the PDF's preprocessing step
data["tags"] = data["tags"].apply(
    lambda tag_list: [str(word).replace(" ", "") for word in tag_list]
)

# Convert the list of tags into one text string
data["tags"] = data["tags"].apply(lambda tag_list: " ".join(tag_list))

display(data.head())
print(data.loc[0, "tags"])


## 2. Text preprocessing

The PDF describes lowercasing, tokenization, stemming and stopword removal. The executed implementation shown in the PDF uses **Porter stemming** and `CountVectorizer(stop_words="english")` (which removes English stopwords during vectorization). The function below preserves that implementation.


In [ ]:
ps = PorterStemmer()

def text_preprocessing(text):
    new_text = []
    for word in str(text).split():
        lower_word = word.lower()
        new_text.append(ps.stem(lower_word))
    return " ".join(new_text)

data["tags"] = data["tags"].apply(text_preprocessing)

display(data.head())
print(data.loc[0, "tags"])


## 3. Bag of Words (CountVectorizer)

The original project uses a maximum vocabulary size of 5,000 features and English stopword removal.


In [ ]:
cv = CountVectorizer(max_features=5000, stop_words="english")
vectors = cv.fit_transform(data["tags"]).toarray()

print("Vector matrix shape:", vectors.shape)
print("Vocabulary size:", len(cv.get_feature_names_out()))


In [ ]:
# Cosine similarity between every pair of movies
similarity = cosine_similarity(vectors)

print("Similarity matrix shape:", similarity.shape)


## 4. Recommendation function

The original notebook returns the five most similar movies, excluding the selected movie itself. This version also handles unknown titles and case differences safely.


In [ ]:
def recommend(movie):
    matches = data.index[data["title"].str.casefold() == str(movie).casefold()].tolist()

    if not matches:
        print(f"Movie '{movie}' was not found. Check the spelling or choose a title from data['title'].")
        return []

    movie_index = matches[0]
    distance = similarity[movie_index]

    movie_list = sorted(
        list(enumerate(distance)),
        reverse=True,
        key=lambda item: item[1]
    )[1:6]

    recommendations = []
    for index, score in movie_list:
        title = data.iloc[index].title
        recommendations.append(title)
        print(f"{title}  (similarity: {score:.4f})")

    return recommendations


In [ ]:
# Sample recommendations shown/tested in the PDF
recommend("Avatar")


In [ ]:
recommend("Iron Man")


In [ ]:
recommend("Batman")


In [ ]:
recommend("Superman")


In [ ]:
recommend("X-Men")


In [ ]:
recommend("Spider-Man")


## 5. Save the trained recommendation data

These files are optional, but useful later when connecting this model to the CINEVERSE website/API. The similarity matrix and movie dataframe are saved separately.


In [ ]:
with open("movie_list.pkl", "wb") as file:
    pickle.dump(data, file)

with open("similarity.pkl", "wb") as file:
    pickle.dump(similarity, file)

with open("count_vectorizer.pkl", "wb") as file:
    pickle.dump(cv, file)

print("Saved: movie_list.pkl, similarity.pkl, count_vectorizer.pkl")


In [ ]:
# Download the generated model files from Colab
from google.colab import files

files.download("movie_list.pkl")
files.download("similarity.pkl")
files.download("count_vectorizer.pkl")


## Notes on reconstruction and corrections

- The PDF is a scanned/exported notebook, so some code characters were visually ambiguous. The notebook reconstructs the executable pipeline represented by the visible code and outputs.
- Corrected obvious OCR/transcription issues such as `Countvectorizer` → `CountVectorizer`, inconsistent quote marks, missing brackets/parentheses, and the misspelled `reconmend` function name.
- The source PDF contains many exploratory display/inspection commands and their outputs (for example, checking `.shape`, `.columns`, individual rows, Python string/list examples, and printing the full vocabulary). Those are not required to execute the recommendation model, so they are represented by concise inspection cells rather than reproducing every printed output.
- The source's stated stopword-removal step is implemented through `CountVectorizer(stop_words="english")`; no separate NLTK stopword-removal function was present in the executed pipeline.
- The notebook requires the two original TMDB CSV datasets. The PDF does not embed those datasets.


## Appendix: exploratory/inspection commands visible in the PDF

These cells reproduce the useful inspection and small Python demonstrations shown between the main pipeline cells. They are optional and do not change the trained recommendation model.


In [ ]:
# Dataset inspection commands shown in the source notebook
display(movies.head(3))
display(credits.head())
print('Movies shape:', movies.shape)
print('Credits shape:', credits.shape)
print('Columns:', movies.columns, credits.columns)
display(movies.head(2))
display(df.head(3))


In [ ]:
# Inspect parsed feature columns and individual examples
display(df['genres'].head())
display(df['keywords'].head())
display(df['cast'].head())
display(df['crew'].head())
print('Overview example:', df['overview'].iloc[0])
print('Tags example:', data['tags'].iloc[0])


In [ ]:
# Small string/list demonstrations shown in the PDF
print(' '.join(['i', 'am', 'a', 'boy']))
print('I AM a Boy'.lower())
print('i am a boy'.replace(' ', ''))
print([5] + [5])
print('love / lover / lovers / lovable / lovely / loving:')
for word in ['love', 'lover', 'lovers', 'lovable', 'lovely', 'loving']:
    print(ps.stem(word))


In [ ]:
# Vectorizer and similarity inspection commands shown in the PDF
print('Vector shape:', vectors.shape)
print('First vocabulary terms:', cv.get_feature_names_out()[:50])
print('Number of vectors:', len(vectors))
print('Similarity matrix size:', len(similarity))
print('First movie similarity row (first 20 values):', similarity[0][:20])
print('Top five similar movie indices/scores for the first movie:')
print(sorted(list(enumerate(similarity[0])), reverse=True, key=lambda item: item[1])[1:6])
